In [36]:
# Handle all the imports of the libraries here.
%matplotlib tk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy as spy
import astropy.io as ast
from sunpy.net import Fido, attrs as a
from glob import glob
from math import asin,cos,sin,pi
from datetime import datetime

In [4]:
'''
This section of code searches and downloads the images data from SDO and downloads and
saves them in the computer locally in the FITS file format.
'''
result = Fido.search(a.Time('2022/05/29','2022/06/08'), a.Instrument.hmi,a.Physobs.intensity,a.Sample(ast.units.hour*24))
files = Fido.fetch(result, path = '../FITS Files/{file}')

AttributeError: module 'astropy.io' has no attribute 'units'

In [37]:
def extract_metadata(filepath):
    '''
    This function reads the FITS file from its path and then fetches the metadata which
    includes the header and data of the file into separate lists giving the 
    coordinates and radius of the Sun.

    Parameters: The string filepath that contains the file path of the FITS file.
    
    Returns: A tuple containing the header and the data of the FITS file.
    '''
    with ast.fits.open(filepath) as hdulist:
        file_header = hdulist[1].header
        file_data = hdulist[1].data
        sun_x_coord = file_header['CRPIX1']
        sun_y_coord = file_header['CRPIX2']
        sun_radius = file_header['RSUN_OBS']
        pixel_arcsecs = file_header['CDELT1']
    return file_header,file_data


In [38]:
def generate_sunspot_coords(file_data):
    '''
    This function fetches the image 2D array and generates an image of the Sun using 
    the matplotlib.imshow function. Finally, it generates a tuple containing the 
    coordinates of the sunspot and returns that tuple.
    Note: This function uses Matplotlib's ginput function where it is required
    to manually click on the position of the sunspot once and then the 
    coordinates are returned in a tuple.

    Parameters: the array file_data of the FITS file.
    
    Returns: the array sunspot_coords which contains the coordinates of the sunspot.
    '''
    cmap_color = 'YlOrBr_r'
    plt.imshow(file_data,cmap=cmap_color)
    sunspot_coords = plt.ginput(n=1,timeout=0)
    plt.close()
    return sunspot_coords[0]

In [39]:
def fetch_latitude_longitude(x,y,sun_x,sun_y,sun_R):
    '''
    This function converts the 2D cartersian coordinates of the form ($x_0$, $y_0$)
    into the 3D coordinates of lattitude and longitude.
    
    The math: $\deltax=x-sun_x$ 
              $\deltay=sun_y-y$
              $\phi=arcsin(\frac{\deltay}{sun_R})$
              $\theta=arcsin(\frac{\deltax}{(sun_R\times\cos(\phi))})$
    

    Parameters: the original x,y coordinates of the sunspot of type float.
                the sun_x,sun_y coordinates of the center of the sun of type float.
                the radius sun_R of the sun of type float.
    
    Returns: phi and theta (lattitude and longitude) coordinates of the sunspot
             as a tuple.
    '''
    delta_x = x - sun_x
    delta_y = sun_y - y
    phi = asin(delta_y/sun_R) # Latitude
    theta = asin(delta_x/(sun_R * cos(phi))) # Longitude

    return phi,theta


<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
C:\Users\aarsh\AppData\Local\Temp\ipykernel_25568\279060806.py:2: SyntaxWarning: invalid escape sequence '\d'
  '''


In [40]:
def calculate_synodic_period(delta_theta, delta_t):
    '''
    This function calculates the synodic time period of rotation of the sun using the
    change in angular displacement over a period of time.

    The math:
    $\Delta\theta=\theta_2-\theta_1$
    $t=t_2-t_1$

    $\omega=\frac{\Delta\theta}{\Deltat}$
    $P_syn=\frac{2\times\pi}{\omega_syn}$
    
    Parameters: $\theta1$, $\theta2$, $t_1$, $t_2$, all of type float.
    Returns: $P_syn$ the synodic time period in days of the sunspot of type float.
    '''
    # angular displacement of the sunspot which is just the change in the angular
    # displacement of the sunspot and the absolute value to handle negative values
    # as the sunspot can move from left to right or right to left.
    delta_theta = abs(delta_theta)
    omega = delta_theta / delta_t # angular synodic velocity/speed of the sunspot
    P_syn = (2*pi) / omega # Synodic time period of the sunspot

    return P_syn

<>:2: SyntaxWarning: invalid escape sequence '\D'
<>:2: SyntaxWarning: invalid escape sequence '\D'
C:\Users\aarsh\AppData\Local\Temp\ipykernel_25568\2732221068.py:2: SyntaxWarning: invalid escape sequence '\D'
  '''


In [41]:
def synodic_to_sidereal(P_syn):
    '''
    This function converts the synodic time period to sidereal time period.

    The math:

    $\frac{1}{P_sid}=\frac{1}{P_syn}+\frac{1}{P_E}$

    Parameters: P_syn: The synodic time period of the sunspot in days of type float.
    Returns: P_sid: The sidereal time period of the sunspot in days of type float.
    '''
    P_E = 365.25 # Earth's orbital period of 365.25 days
    P_sid = (P_syn * P_E) / (P_syn + P_E) # The sidereal time of the rotation of the sun.

    return P_sid # return the sidereal time period in radians per day
    

In [16]:
def main():
    '''
    This is the main function that loops over the FITS files and calls the functions
    implemented above to fetch the required data and then perform the required analysis.
    It exports the .csv file and also returns the sunspot_data.

    Parameters: None
    Returns: sunspot_data of type list which contains the data of the sunspot.
    '''
    files = sorted(glob("../FITS Files/*.FITS"))
    sunspot_data = []

    for file_path in files:
        file_header,file_data = extract_metadata(file_path)
        x_click, y_click = generate_sunspot_coords(file_data)
        date_str = file_header['DATE-OBS']
        date_obs = datetime.fromisoformat(date_str)        
        sun_center_x = file_header['CRPIX1']
        sun_center_y = file_header['CRPIX2']
        sun_radius = file_header['RSUN_OBS']
        pixel_arcsecs = file_header['CDELT1']
        sun_radius_pixels = sun_radius/pixel_arcsecs

        phi, theta = fetch_latitude_longitude(x_click,y_click,sun_center_x,sun_center_y,sun_radius_pixels)

        current_file_data = {"Date": date_obs, "Latitude": phi, "Longitude" : theta, 
                             "radius" : sun_radius_pixels}

        sunspot_data.append(current_file_data)

    df = pd.DataFrame(sunspot_data)
    df.to_csv("sunspot_raw_data.csv",index=False)

    return sunspot_data




In [17]:
main()

2026-03-13 13:30:35 - matplotlib.figure - INFO: input 1: 2856.504329, 3051.551948
2026-03-13 13:30:42 - matplotlib.figure - INFO: input 1: 2535.119048, 3051.551948
2026-03-13 13:30:49 - matplotlib.figure - INFO: input 1: 2202.651515, 3062.634199
2026-03-13 13:30:56 - matplotlib.figure - INFO: input 1: 1870.183983, 3073.716450
2026-03-13 13:31:02 - matplotlib.figure - INFO: input 1: 1526.634199, 3084.798701
2026-03-13 13:31:08 - matplotlib.figure - INFO: input 1: 1238.495671, 3095.880952
2026-03-13 13:31:14 - matplotlib.figure - INFO: input 1: 961.439394, 3084.798701
2026-03-13 13:31:22 - matplotlib.figure - INFO: input 1: 750.876623, 3095.880952
2026-03-13 13:31:28 - matplotlib.figure - INFO: input 1: 573.560606, 3118.045455


[{'Date': datetime.datetime(2022, 5, 29, 0, 0, 37, 200000),
  'Latitude': -0.5603946243781157,
  'Longitude': 0.5402759670415395,
  'radius': 1878.782681758211},
 {'Date': datetime.datetime(2022, 5, 30, 0, 0, 37, 300000),
  'Latitude': -0.5603802021138335,
  'Longitude': 0.3178157209024343,
  'radius': 1878.47401470136},
 {'Date': datetime.datetime(2022, 5, 31, 0, 0, 37, 300000),
  'Latitude': -0.5674912464430586,
  'Longitude': 0.10421671458751545,
  'radius': 1878.1751584711378},
 {'Date': datetime.datetime(2022, 6, 1, 0, 0, 37, 400000),
  'Latitude': -0.5745545813832829,
  'Longitude': -0.1064795197233997,
  'radius': 1877.8861090995663},
 {'Date': datetime.datetime(2022, 6, 2, 0, 0, 37, 500000),
  'Latitude': -0.5816617073072698,
  'Longitude': -0.33187086862086473,
  'radius': 1877.6104727062063},
 {'Date': datetime.datetime(2022, 6, 3, 0, 0, 37, 600000),
  'Latitude': -0.5888314638087462,
  'Longitude': -0.537460764911309,
  'radius': 1877.3332261981284},
 {'Date': datetime.datet

In [45]:
def differential_rotation_arrays(file_path):
    omegas = []
    sin_square_phi = []

    df = pd.read_csv(file_path)
    raw_dates = df['Date']
    dates = []
    for i in range(len((raw_dates))):
        date = pd.to_datetime(raw_dates[i])
        dates.append(date)
    phi = df['Latitude']
    theta = df['Longitude']
    delta_t = []
    delta_theta = []

    for i in range(len(dates)-1):
        current_delta_t = dates[i+1] - dates[i]
        seconds = current_delta_t.total_seconds()
        days = seconds/86400
        current_delta_theta = abs(theta[i+1] - theta[i])

        if days <= 0:
            continue
        
        days = 1
        delta_t.append(days)
        delta_theta.append(current_delta_theta)
        synodic_period = calculate_synodic_period(current_delta_theta,days)
        sidereal_period = synodic_to_sidereal(synodic_period)
        current_omega = (2 * pi)/(sidereal_period)
        omegas.append(current_omega)
        current_sin_square_phi = sin(phi[i])**2
        sin_square_phi.append(current_sin_square_phi)

    return omegas, sin_square_phi

In [ ]:
def final_graph(x_array,y_array):
    omegas = np.array(y_array)
    sin_square_phi = np.array(x_array)
    plt.scatter(sin_square_phi, omegas)
    plt.title("$\omega(\phi)=A+B\sin^2(\phi)$")
    plt.xlabel("$\sin^2(\phi)$")
    plt.ylabel("$\omega(\phi)$")
    coeffs, uncertaintity = np.polyfit(sin_square_phi,omegas,1, cov=True)
    slope = coeffs[0]
    intercept = coeffs[1]
    linear_fit = slope * sin_square_phi + intercept
    slope_uncer = np.sqrt(uncertaintity[0,0])
    intercept_uncer = np.sqrt(uncertaintity[1,1])
    B = slope *(180/pi)
    A_uncer = intercept_uncer *(180/pi)
    B_uncer = slope_uncer *(180/pi)
    A = intercept * (180/pi)
    print(f"The slope of the graph is B = {slope} $\pm$ {slope_uncer} and the y-intercept is A = {intercept} $\pm$ {intercept_uncer}.")
    print(f"The value of A is {A} $\pm$ {A_uncer} degrees/day.")
    print(f"The value of B is {B} $\pm$ {B_uncer} degrees/day.")
    plt.plot(sin_square_phi,linear_fit,color='red')
    plt.savefig("linear_regression.jpeg")
    plt.show()



<>:5: SyntaxWarning: invalid escape sequence '\o'
<>:6: SyntaxWarning: invalid escape sequence '\s'
<>:7: SyntaxWarning: invalid escape sequence '\o'
<>:18: SyntaxWarning: invalid escape sequence '\p'
<>:18: SyntaxWarning: invalid escape sequence '\p'
<>:19: SyntaxWarning: invalid escape sequence '\p'
<>:20: SyntaxWarning: invalid escape sequence '\p'
<>:5: SyntaxWarning: invalid escape sequence '\o'
<>:6: SyntaxWarning: invalid escape sequence '\s'
<>:7: SyntaxWarning: invalid escape sequence '\o'
<>:18: SyntaxWarning: invalid escape sequence '\p'
<>:18: SyntaxWarning: invalid escape sequence '\p'
<>:19: SyntaxWarning: invalid escape sequence '\p'
<>:20: SyntaxWarning: invalid escape sequence '\p'
C:\Users\aarsh\AppData\Local\Temp\ipykernel_25568\4045434960.py:5: SyntaxWarning: invalid escape sequence '\o'
  plt.title("$\omega(\phi)=A+B\sin^2(\phi)$")
C:\Users\aarsh\AppData\Local\Temp\ipykernel_25568\4045434960.py:6: SyntaxWarning: invalid escape sequence '\s'
  plt.xlabel("$\sin^2(\p

In [56]:
diff_rot_tuples = differential_rotation_arrays('sunspot_raw_data.csv')
omegas = diff_rot_tuples[0]
sin_square_phi = diff_rot_tuples[1]
final_graph(sin_square_phi,omegas)

The slope of the graph is B = -0.08006821506002665 $\pm$ 0.02546103948600171 and the y-intercept is A = 0.256377734038388 $\pm$ 0.005138805854605645.
The value of A is 14.68936212152714 $\pm$ 0.2944318872060216 degrees/radian.
The value of B is -4.587570796085345 $\pm$ 1.4588101045638369 degrees/radian.
